In [1]:
import numpy as np
import pandas as pd
import torch
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── konfiguracja ──────────────────────────────────────────────────────────────
MC_ELEC_FILE = "../../../Data_Separated/MC/electron.csv"
MC_MUON_FILE = "../../../Data_Separated/MC/muon.csv"
MODEL_PATH   = "../../klasyfikator/trained/model_trained_on_filtered.pt"
SCALER_PATH  = "../../klasyfikator/trained/scaler_mlp2.pkl"

THRESH_MUON   = 0.4
THRESH_ELEC   = 0.6
MAX_PER_CLASS = 100000

FEATURE_COLUMNS = [
    "track_TRTHits", "track_PixelHits", "track_SCTHits", "track_PixeldEdX",
    "topo_cluster_eta", "topo_cluster_phi", "topo_cluster_EM_prob",
    "topo_cluster_pt", "pt_ratio", "topo_cluster_lambda", "topo_cluster_lambda2"
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ── wczytanie i filtracja ─────────────────────────────────────────────────────
print("Wczytuję MC...")
df_el = pd.read_csv(MC_ELEC_FILE)
df_mu = pd.read_csv(MC_MUON_FILE)

FILTER = lambda df: df[
    (df['topo_cluster_eta']    != -10.0) &
    (df['topo_cluster_pt']     !=  -1.0) &
    (df['topo_cluster_phi']    != -10.0) &
    (df['topo_cluster_EM_prob']!=  -1.0) &
    (df['pt_ratio']            !=  -1.0) &
    (df['topo_cluster_lambda'] !=  -1.0) &
    (df['topo_cluster_lambda2']!=  -1.0)
].reset_index(drop=True)

df_el = FILTER(df_el).iloc[:MAX_PER_CLASS]
df_mu = FILTER(df_mu).iloc[:MAX_PER_CLASS]

print(f"Elektrony MC: {len(df_el)}")
print(f"Miony MC:     {len(df_mu)}")

# ── inferencja ────────────────────────────────────────────────────────────────
print("Ładuję model i scaler...")
scaler = joblib.load(SCALER_PATH)
model  = torch.jit.load(MODEL_PATH, map_location=DEVICE)
model.eval()

def get_prob(df):
    X = scaler.transform(df[FEATURE_COLUMNS].values.astype(np.float32))
    t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        return torch.sigmoid(model(t)[:, 0]).cpu().numpy()

print("Inferencja...")
prob_el = get_prob(df_el)
prob_mu = get_prob(df_mu)

# accuracy według MC truth
acc_el = (prob_el > THRESH_ELEC).sum() / len(prob_el)
acc_mu = (prob_mu < THRESH_MUON).sum() / len(prob_mu)
dead_el = ((prob_el >= THRESH_MUON) & (prob_el <= THRESH_ELEC)).sum()
dead_mu = ((prob_mu >= THRESH_MUON) & (prob_mu <= THRESH_ELEC)).sum()

print(f"\nAccuracy elektrony: {acc_el*100:.2f}%  (martwa strefa: {dead_el})")
print(f"Accuracy miony:     {acc_mu*100:.2f}%  (martwa strefa: {dead_mu})")

# ── rysowanie ─────────────────────────────────────────────────────────────────
COLOR_MU = "#3A7FBF"
COLOR_EL = "#E05C3A"
BINS     = 80

def norm_hist(ax, mu_data, el_data, xlabel, title, xrange):
    kw = dict(bins=BINS, range=xrange, histtype="step", linewidth=1.8, density=True)
    ax.hist(mu_data, color=COLOR_MU, label="Miony (MC truth)",    **kw)
    ax.hist(el_data, color=COLOR_EL, label="Elektrony (MC truth)", **kw)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel("Znorm. liczba zdarzeń", fontsize=9)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_xlim(xrange)

fig = plt.figure(figsize=(14, 10))
fig.suptitle(
    f"Model 2 — separacja e/μ na danych MC\n"
    f"Accuracy:  elektron {acc_el*100:.2f}%   muon {acc_mu*100:.2f}%   "
    f"(próg e > {THRESH_ELEC}, μ < {THRESH_MUON})",
    fontsize=13, y=0.99
)

gs = gridspec.GridSpec(2, 2, hspace=0.42, wspace=0.32)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

# 1. Masa niezmiennicza
norm_hist(ax1,
          df_mu["inv_mass"].values, df_el["inv_mass"].values,
          xlabel="Masa niezmiennicza [GeV]",
          title="Masa niezmiennicza pary",
          xrange=(2.5, 3.5))

# 2. pT wybranego tracka
norm_hist(ax2,
          df_mu["track_pt_sel"].values, df_el["track_pt_sel"].values,
          xlabel="$p_T$ tracka [GeV]",
          title="$p_T$ wybranego tracka",
          xrange=(0.0, 5.0))

# 3. EM probability
norm_hist(ax3,
          df_mu["topo_cluster_EM_prob"].values, df_el["topo_cluster_EM_prob"].values,
          xlabel="EM probability",
          title="EM probability (topo cluster)",
          xrange=(0.0, 1.0))

# 4. eta wybranego tracka
norm_hist(ax4,
          df_mu["track_eta_sel"].values, df_el["track_eta_sel"].values,
          xlabel="η tracka",
          title="η wybranego tracka",
          xrange=(-2.5, 2.5))

plt.savefig("plot_mlp2_mc.png", dpi=150, bbox_inches="tight")
plt.show()
print("Zapisano: plot_mlp2_mc.png")

Device: cuda
Wczytuję MC...


FileNotFoundError: [Errno 2] No such file or directory: '../../Data_Separated/MC/electron.csv'